### Condition-organ

In [9]:
# ============================================================
# DOMAIN SHIFT (LINEAR PROBE ONLY) — ROW-EMBEDDINGS (ALREADY SAVED)
#
# ✅ Your constraint:
#   - embeddings are ALREADY saved (e.g., msi_chan_only__REP.npy, fused_all__REP.npy, etc.)
#   - this script ONLY LOADS them and runs domain-shift linear probes.
#
# ✅ Backend (single unified):
#   embeddings: (N_rows, D)  npy
#   row_ids:    (N_rows,)    npy  -> indices into channels_with_candidates.parquet rows
#
# Works with any saved set in:
#   spatial_metabolomics_atlas/channel_embeddings/
#
# Examples:
#   - msi_chan_only__REP.npy         + row_ids__msi_chan_only__REP.npy
#   - fused_tile__REP.npy            + row_ids__tile__REP.npy
#   - fused_chan__REP.npy            + row_ids__chan__REP.npy
#   - fused_all__REP.npy             + row_ids__all__REP.npy
#   - fused_msi__REP.npy             + row_ids__msi__REP.npy
#
# Train on DOMAIN=A datasets, test on DOMAIN=B datasets.
# Inside TRAIN domain only: dataset-grouped TRAIN/VAL split (no leakage).
#
# Includes:
#   - Explicit DOMAIN_VALUES (no auto top-2)
#   - analyzerType (3 values): all pairwise shifts both directions
#   - Organism_Part restricted to TOPK_ORGANS=6 by majority (dataset-count)
#   - FAST LP: SGDClassifier (logistic loss) + dataset-coded masks
#   - tqdm where it matters
#   - optional dataset-balanced row caps for trainpool/test
# ============================================================

from __future__ import annotations

from pathlib import Path
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning)
warnings.filterwarnings("ignore", message=".*converge.*", category=ConvergenceWarning)
warnings.filterwarnings("ignore", module="sklearn")


# ============================================================
# PATHS
# ============================================================
BASE = Path("metaspace_images_dump")
CHAN_PQ = BASE / "channels_with_candidates.parquet"

MSI_DIR = Path("msi-vitmae_REBUILT2_2/embeddings")
MSI_META_CSV = MSI_DIR / "meta.csv"  # dataset-level labels live here

ATLAS = Path("spatial_metabolomics_atlas")
EMB_DIR = ATLAS / "channel_embeddings"

OUT_ROOT = ATLAS / "benchmarks" / "domain_shift_single_channel"
OUT_ROOT.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 6740

TRAIN_FRAC = 0.80
VAL_FRAC   = 0.20
assert abs((TRAIN_FRAC + VAL_FRAC) - 1.0) < 1e-9

SPLIT_REPEATS = 3
SPLIT_SEEDS = [SEED + 1000 * r for r in range(SPLIT_REPEATS)]

# Linear probe hyperparams
C_GRID = [1.0]              # keep small for speed; expand later if needed
MAX_EPOCHS_BIN   = 2000
MAX_EPOCHS_MULTI = 2000

TASKS = ["Condition", "Organism_Part"]
TOPK_ORGANS = 6

# Explicit domain values (lowercased later)
DOMAIN_VALUES = {
    "polarity": ["positive", "negative"],
    "organism": ["mus musculus", "homo sapiens"],
    "ionisationSource": ["DESI", "MALDI"],
    "analyzerType": ["FTICR", "Orbitrap", "timsTOF"],
}
DOMAIN_COLS = list(DOMAIN_VALUES.keys())

TUMOR_LABELS  = {"Tumor", "Cancer"}
NORMAL_LABELS = {"Wildtype", "Healthy"}

DATASET_COL = "dataset_id"
TILE_R_COL  = "tile_r"
TILE_C_COL  = "tile_c"
CHAN_IDX_COL = "channel_idx"

UNKNOWN_STR = {"", "nan", "None", "none", "NaN"}

# Optional caps (dataset-balanced)
MAX_TRAINPOOL_ROWS = 30000
MAX_TEST_ROWS      = None

# ============================================================
# EMBEDDING SETS TO RUN (EDIT THIS)
# ============================================================
REPS = ["PCA", "MAE-RandInit", "MAE-ImageNet", "MAE-MSI"]

# Which saved embedding variants do you want to compare?
# Choose any subset from: ["msi_chan_only","fused_tile","fused_chan","fused_all","fused_msi"]
EMB_SETS = ["fused_all"]

# Map emb_set -> (emb_file_prefix, row_ids_file_prefix)
# (row_ids names are from your saver)
EMB_SET_TO_FILES = {
    "msi_chan_only": ("msi_chan_only", "row_ids__msi_chan_only"),
    "fused_tile":    ("fused_tile",    "row_ids__tile"),
    "fused_chan":    ("fused_chan",    "row_ids__chan"),
    "fused_all":     ("fused_all",     "row_ids__all"),
    "fused_msi":     ("fused_msi",     "row_ids__msi"),
}


# ============================================================
# Helpers
# ============================================================
def safe_str(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def norm_str(x) -> str:
    return safe_str(x).strip().lower()

def make_condition_binary(cond: pd.Series) -> np.ndarray:
    s = cond.astype(str)
    y = s.map(lambda x: 1 if x in TUMOR_LABELS else (0 if x in NORMAL_LABELS else np.nan))
    return y.values

def stratified_group_split_train_val_dataset_ids(ds_ids, ds_labels, seed, val_frac):
    ds_ids = np.asarray(ds_ids).astype(str)
    ds_labels = np.asarray(ds_labels)

    if len(ds_ids) != len(ds_labels):
        raise ValueError("ds_ids and ds_labels must have same length")
    if len(np.unique(ds_labels)) < 2:
        raise ValueError("Need >=2 classes at dataset level for stratified split")

    vc = pd.Series(ds_labels).value_counts()
    if int(vc.min()) < 2:
        raise ValueError("Some dataset-level class has <2 datasets (cannot stratify).")

    idx_all = np.arange(len(ds_ids))
    sss = StratifiedShuffleSplit(n_splits=1, test_size=float(val_frac), random_state=int(seed))
    idx_tr, idx_va = next(sss.split(idx_all, ds_labels))
    return ds_ids[idx_tr], ds_ids[idx_va]

def fast_standardize_train(Xtr: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    mu = Xtr.mean(axis=0, dtype=np.float64)
    sd = Xtr.std(axis=0, dtype=np.float64)
    sd[sd == 0] = 1.0
    return mu, sd

def apply_standardize(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    return (X - mu) / sd

def filter_task_classes_in_train_domain(meta_train: pd.DataFrame, task_label_col: str, min_ds_per_class: int = 2) -> pd.DataFrame:
    vc = meta_train[task_label_col].value_counts()
    keep = vc[vc >= int(min_ds_per_class)].index.astype(str).tolist()
    return meta_train[meta_train[task_label_col].isin(keep)].reset_index(drop=True)

def cap_rows_by_dataset_codes(idx_rows: np.ndarray, ds_codes: np.ndarray, max_total: int | None, seed: int) -> np.ndarray:
    idx_rows = np.asarray(idx_rows, dtype=np.int64)
    if max_total is None or len(idx_rows) <= int(max_total):
        return idx_rows

    rng = np.random.default_rng(int(seed))
    codes = ds_codes[idx_rows].astype(np.int32, copy=False)

    uniq = np.unique(codes)
    rng.shuffle(uniq)

    base = int(max_total) // max(len(uniq), 1)
    rem  = int(max_total) - base * len(uniq)

    kept = []
    for code in uniq:
        arr = idx_rows[codes == code].copy()
        rng.shuffle(arr)
        take = base + (1 if rem > 0 else 0)
        if rem > 0:
            rem -= 1
        kept.append(arr[:min(take, len(arr))])

    out = np.concatenate(kept) if kept else idx_rows[:int(max_total)]
    rng.shuffle(out)
    return out[:int(max_total)]

def _alpha_from_C(C: float) -> float:
    C = float(C)
    if C <= 0:
        C = 1.0
    return 1.0 / C

def fit_sgd_logreg_scaled(Xtr_s, ytr, Xte_s, yte, C, seed, max_epochs):
    clf = SGDClassifier(
        loss="log_loss",
        alpha=_alpha_from_C(C),
        penalty="l2",
        max_iter=int(max_epochs),
        tol=1e-3,
        n_jobs=-1,
        class_weight="balanced",
        random_state=int(seed),
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=5,
    )
    clf.fit(Xtr_s, ytr)
    yhat = clf.predict(Xte_s)
    return {
        "acc": float(accuracy_score(yte, yhat)),
        "f1_macro": float(f1_score(yte, yhat, average="macro")),
    }

def select_C_on_val_scaled(Xtr_s, ytr, Xva_s, yva, C_grid, seed, max_epochs):
    best_C, best_f1 = None, -1.0
    for C in C_grid:
        res = fit_sgd_logreg_scaled(Xtr_s, ytr, Xva_s, yva, C=C, seed=seed, max_epochs=max_epochs)
        if res["f1_macro"] > best_f1:
            best_f1 = res["f1_macro"]
            best_C = C
    return float(best_C)

def _slug(s: str) -> str:
    s = str(s)
    out = []
    for ch in s:
        out.append(ch if ch.isalnum() else "_")
    out = "".join(out)
    while "__" in out:
        out = out.replace("__", "_")
    out = out.strip("_")
    return out[:80] if out else "x"

def generate_domain_pairs(values: list[str]) -> list[tuple[str, str]]:
    pairs = []
    for a, b in itertools.combinations(values, 2):
        pairs.append((a, b))
        pairs.append((b, a))
    return pairs

def restrict_to_topk_organs(meta_task: pd.DataFrame, *, k: int) -> pd.DataFrame:
    if k is None or k <= 0:
        return meta_task
    vc = meta_task["task_label"].value_counts()
    keep = vc.head(int(k)).index.astype(str).tolist()
    return meta_task[meta_task["task_label"].isin(keep)].reset_index(drop=True)

def plot_domain_shift_by_task(summary_df: pd.DataFrame, *, task: str, domain_col: str, emb_set: str, out_png: Path):
    sub = summary_df[(summary_df["task"] == task) & (summary_df["domain_col"] == domain_col) & (summary_df["emb_set"] == emb_set)].copy()
    if sub.empty:
        return

    dirs = (
        sub[["train_domain_value", "test_domain_value"]]
        .drop_duplicates()
        .apply(lambda r: f"{r['train_domain_value']} → {r['test_domain_value']}", axis=1)
        .tolist()
    )
    reps = REPS

    mean = {(r, d): np.nan for r in reps for d in dirs}
    std  = {(r, d): 0.0   for r in reps for d in dirs}

    for _, rr in sub.iterrows():
        dlab = f"{rr['train_domain_value']} → {rr['test_domain_value']}"
        mean[(str(rr["rep"]), dlab)] = float(rr["mean_f1"])
        std[(str(rr["rep"]), dlab)]  = float(rr["std_f1"]) if np.isfinite(rr["std_f1"]) else 0.0

    x = np.arange(len(dirs))
    width = 0.8 / max(len(reps), 1)

    plt.figure(figsize=(max(10.5, 1.8 * len(dirs)), 4.4))
    for i, rep in enumerate(reps):
        vals = [mean[(rep, d)] for d in dirs]
        errs = [std[(rep, d)] for d in dirs]
        xpos = x - 0.4 + (i + 0.5) * width
        plt.bar(xpos, vals, width=width, label=rep)
        plt.errorbar(xpos, vals, yerr=errs, fmt="none", capsize=3, c="black", lw=1)

    plt.xticks(x, dirs, rotation=0)
    plt.ylabel("Test Macro-F1 (mean ± std)")
    plt.title(f"Domain shift — {task}\nDOMAIN_COL={domain_col} | EMB={emb_set}")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.savefig(out_png, dpi=300)
    plt.close()
    print("[OK] Saved plot:", out_png)


# ============================================================
# LOAD SAVED EMBEDDINGS (NO SAVING HERE)
# ============================================================
def load_saved_row_embeddings(rep: str, emb_set: str) -> tuple[np.ndarray, np.ndarray]:
    if emb_set not in EMB_SET_TO_FILES:
        raise KeyError(f"Unknown emb_set={emb_set}. Valid: {sorted(EMB_SET_TO_FILES.keys())}")

    emb_prefix, rid_prefix = EMB_SET_TO_FILES[emb_set]
    emb_p = EMB_DIR / f"{emb_prefix}__{rep}.npy"
    rid_p = EMB_DIR / f"{rid_prefix}__{rep}.npy"

    if not emb_p.exists():
        raise FileNotFoundError(f"Missing embeddings: {emb_p}")
    if not rid_p.exists():
        raise FileNotFoundError(f"Missing row_ids: {rid_p}")

    X = np.load(emb_p, mmap_mode="r")         # (N,D)
    row_ids = np.load(rid_p)                  # (N,)
    if X.ndim != 2:
        raise RuntimeError(f"{emb_p.name}: expected 2D (N,D) got {X.shape}")
    if row_ids.ndim != 1 or len(row_ids) != len(X):
        raise RuntimeError(f"{rid_p.name}: expected (N,) with same N as X. got {row_ids.shape}, X N={len(X)}")

    return np.asarray(X, dtype=np.float32, order="C"), row_ids.astype(np.int64, copy=False)


# ============================================================
# RUN ONE DOMAIN COL (ALL PAIRS) — per emb_set
# ============================================================
def run_one_domain_col(
    *,
    domain_col: str,
    ch_df_full: pd.DataFrame,
    meta_ds_all: pd.DataFrame,
):
    OUT_BASE = OUT_ROOT / f"domain_{_slug(domain_col)}"
    OUT_BASE.mkdir(parents=True, exist_ok=True)

    domain_pairs = generate_domain_pairs(DOMAIN_VALUES[domain_col])

    for domA, domB in domain_pairs:
        print(f"\n[DOMAIN] DOMAIN_COL={domain_col}  TRAIN='{domA}'  TEST='{domB}'")

        meta_ds = meta_ds_all[meta_ds_all[domain_col].isin({domA, domB})].copy().reset_index(drop=True)
        if meta_ds.empty:
            print(f"[SKIP] {domain_col}: no datasets for {domA}/{domB}")
            continue

        meta_ds["domain"] = meta_ds[domain_col].map(lambda x: 0 if x == domA else 1).astype(np.int64)
        ds_ok = set(meta_ds[DATASET_COL].astype(str).tolist())

        for emb_set in EMB_SETS:
            OUT_PAIR = OUT_BASE / f"{_slug(domA)}__to__{_slug(domB)}" / emb_set
            OUT_PAIR.mkdir(parents=True, exist_ok=True)
            OUT_PLOT = OUT_PAIR / "plots"
            OUT_PLOT.mkdir(parents=True, exist_ok=True)

            RESULTS_CSV = OUT_PAIR / "results_all.csv"
            SUMMARY_CSV = OUT_PAIR / "summary_meanstd.csv"

            # Load saved embeddings for each rep (and restrict to domA/domB datasets)
            rep_data = {}
            for rep in tqdm(REPS, desc=f"Load reps ({domain_col} {domA}->{domB} | {emb_set})"):
                X, row_ids = load_saved_row_embeddings(rep, emb_set)

                ds_per_row = ch_df_full.iloc[row_ids][DATASET_COL].astype(str).values
                m = np.isin(ds_per_row, np.array(list(ds_ok), dtype=object))

                X = X[m]
                row_ids = row_ids[m]
                ds_per_row = ds_per_row[m]

                print(f"[INFO] rep={rep:12s} emb_set={emb_set:14s} rows in domA/domB: {len(row_ids):,}")
                rep_data[rep] = {"X": X, "ds_per_row": ds_per_row}

            results = []
            split_cache: dict[tuple[str, str, int, str, str], tuple[set[str], set[str]]] = {}

            for rep in tqdm(REPS, desc=f"Linear probe reps ({domain_col} {domA}->{domB} | {emb_set})"):
                X_rep = rep_data[rep]["X"]
                ds_per_row = np.asarray(rep_data[rep]["ds_per_row"], dtype=object)
                if len(X_rep) == 0:
                    continue

                ds_codes, ds_uniques = pd.factorize(ds_per_row, sort=False)
                ds_codes = ds_codes.astype(np.int32, copy=False)
                dsid_to_code = {str(dsid): int(i) for i, dsid in enumerate(ds_uniques)}

                for task in tqdm(TASKS, desc=f"Tasks ({domain_col}/{domA}->{domB}/{emb_set}/{rep})", leave=False):
                    meta_task = meta_ds[[DATASET_COL, "domain", task]].copy()
                    meta_task[task] = meta_task[task].map(safe_str).astype(str)
                    meta_task = meta_task[~meta_task[task].isin(UNKNOWN_STR) & (meta_task[task] != "")].reset_index(drop=True)
                    if meta_task.empty:
                        continue

                    if task == "Condition":
                        y_bin = make_condition_binary(meta_task["Condition"])
                        meta_task = meta_task.loc[~pd.isna(y_bin)].reset_index(drop=True)
                        meta_task["task_label"] = y_bin[~pd.isna(y_bin)].astype(np.int64).astype(str)
                    else:
                        meta_task["task_label"] = meta_task[task].astype(str)

                    if task == "Organism_Part" and TOPK_ORGANS is not None:
                        meta_task = restrict_to_topk_organs(meta_task, k=TOPK_ORGANS)
                        if meta_task.empty or meta_task["task_label"].nunique() < 2:
                            continue

                    meta_train = meta_task[meta_task["domain"] == 0].copy().reset_index(drop=True)
                    meta_test  = meta_task[meta_task["domain"] == 1].copy().reset_index(drop=True)
                    if meta_train.empty or meta_test.empty:
                        continue

                    meta_train = filter_task_classes_in_train_domain(meta_train, "task_label", min_ds_per_class=2)
                    if meta_train["task_label"].nunique() < 2:
                        continue

                    vc = meta_train["task_label"].value_counts()
                    cats = sorted(vc.index.astype(str).tolist(), key=lambda c: (-int(vc.get(c, 0)), str(c)))

                    meta_train["y_ds"] = pd.Categorical(meta_train["task_label"], categories=cats, ordered=True).codes.astype(np.int64)
                    meta_test["y_ds"]  = pd.Categorical(meta_test["task_label"],  categories=cats, ordered=True).codes.astype(np.int64)
                    meta_test = meta_test[meta_test["y_ds"] >= 0].reset_index(drop=True)
                    if meta_test["y_ds"].nunique() < 2:
                        continue

                    ds_train_ids = meta_train[DATASET_COL].astype(str).values
                    y_ds_train   = meta_train["y_ds"].astype(np.int64).values

                    ds2y_train = dict(zip(meta_train[DATASET_COL].astype(str), meta_train["y_ds"].astype(int)))
                    ds2y_test  = dict(zip(meta_test[DATASET_COL].astype(str),  meta_test["y_ds"].astype(int)))

                    train_codes = np.array([dsid_to_code[str(x)] for x in ds_train_ids if str(x) in dsid_to_code], dtype=np.int32)
                    test_codes  = np.array([dsid_to_code[str(x)] for x in meta_test[DATASET_COL].astype(str).tolist() if str(x) in dsid_to_code], dtype=np.int32)

                    idx_row_trainpool = np.where(np.isin(ds_codes, train_codes))[0].astype(np.int64)
                    idx_row_test      = np.where(np.isin(ds_codes, test_codes))[0].astype(np.int64)
                    if len(idx_row_trainpool) < 200 or len(idx_row_test) < 200:
                        continue

                    y_by_code_train = np.full(len(ds_uniques), -1, dtype=np.int64)
                    y_by_code_test  = np.full(len(ds_uniques), -1, dtype=np.int64)
                    for dsid, yy in ds2y_train.items():
                        if dsid in dsid_to_code:
                            y_by_code_train[dsid_to_code[dsid]] = int(yy)
                    for dsid, yy in ds2y_test.items():
                        if dsid in dsid_to_code:
                            y_by_code_test[dsid_to_code[dsid]] = int(yy)

                    y_trainpool = y_by_code_train[ds_codes[idx_row_trainpool]]
                    y_test      = y_by_code_test[ds_codes[idx_row_test]]

                    ok_tr = y_trainpool >= 0
                    ok_te = y_test >= 0
                    idx_row_trainpool = idx_row_trainpool[ok_tr]
                    y_trainpool = y_trainpool[ok_tr]
                    idx_row_test = idx_row_test[ok_te]
                    y_test = y_test[ok_te]
                    if len(np.unique(y_trainpool)) < 2 or len(np.unique(y_test)) < 2:
                        continue

                    idx_row_trainpool = cap_rows_by_dataset_codes(
                        idx_row_trainpool, ds_codes, MAX_TRAINPOOL_ROWS,
                        seed=int(SEED) + 17 + (hash(task + rep + domain_col + domA + domB + emb_set) % 997),
                    )
                    idx_row_test = cap_rows_by_dataset_codes(
                        idx_row_test, ds_codes, MAX_TEST_ROWS,
                        seed=int(SEED) + 29 + (hash("TEST" + task + rep + domain_col + domA + domB + emb_set) % 997),
                    )

                    y_trainpool = y_by_code_train[ds_codes[idx_row_trainpool]]
                    y_test      = y_by_code_test[ds_codes[idx_row_test]]

                    ok_tr = y_trainpool >= 0
                    ok_te = y_test >= 0
                    idx_row_trainpool = idx_row_trainpool[ok_tr]
                    y_trainpool = y_trainpool[ok_tr]
                    idx_row_test = idx_row_test[ok_te]
                    y_test = y_test[ok_te]
                    if len(np.unique(y_trainpool)) < 2 or len(np.unique(y_test)) < 2:
                        continue

                    ds_trainpool_codes_per_row = ds_codes[idx_row_trainpool]

                    for split_seed in SPLIT_SEEDS:
                        cache_key = (domain_col, task, int(split_seed), domA, domB)
                        if cache_key in split_cache:
                            ds_tr_set, ds_va_set = split_cache[cache_key]
                        else:
                            try:
                                ds_tr, ds_va = stratified_group_split_train_val_dataset_ids(
                                    ds_ids=ds_train_ids,
                                    ds_labels=y_ds_train,
                                    seed=int(split_seed) + 101 + (hash(domain_col + task + domA + domB) % 997),
                                    val_frac=VAL_FRAC,
                                )
                            except Exception:
                                continue
                            ds_tr_set = set(map(str, ds_tr))
                            ds_va_set = set(map(str, ds_va))
                            split_cache[cache_key] = (ds_tr_set, ds_va_set)

                        ds_tr_codes = np.array([dsid_to_code[x] for x in ds_tr_set if x in dsid_to_code], dtype=np.int32)
                        ds_va_codes = np.array([dsid_to_code[x] for x in ds_va_set if x in dsid_to_code], dtype=np.int32)

                        m_tr = np.isin(ds_trainpool_codes_per_row, ds_tr_codes)
                        m_va = np.isin(ds_trainpool_codes_per_row, ds_va_codes)

                        tr_idx = np.where(m_tr)[0]
                        va_idx = np.where(m_va)[0]
                        if len(tr_idx) < 50 or len(va_idx) < 20:
                            continue

                        Xtr = X_rep[idx_row_trainpool[tr_idx]]
                        ytr = y_trainpool[tr_idx]
                        Xva = X_rep[idx_row_trainpool[va_idx]]
                        yva = y_trainpool[va_idx]
                        Xte = X_rep[idx_row_test]
                        yte = y_test

                        n_classes = int(len(np.unique(np.concatenate([ytr, yva, yte]))))
                        max_epochs = MAX_EPOCHS_BIN if n_classes == 2 else MAX_EPOCHS_MULTI

                        mu, sd = fast_standardize_train(Xtr)
                        Xtr_s = apply_standardize(Xtr, mu, sd)
                        Xva_s = apply_standardize(Xva, mu, sd)
                        Xte_s = apply_standardize(Xte, mu, sd)

                        best_C = select_C_on_val_scaled(
                            Xtr_s, ytr, Xva_s, yva,
                            C_grid=C_GRID,
                            seed=int(split_seed) + 13 + (hash(task + rep + domain_col + domA + domB + emb_set) % 997),
                            max_epochs=max_epochs,
                        )
                        te = fit_sgd_logreg_scaled(
                            Xtr_s, ytr, Xte_s, yte,
                            C=best_C,
                            seed=int(split_seed) + 77 + (hash(rep + domain_col + domA + domB + emb_set) % 997),
                            max_epochs=max_epochs,
                        )

                        results.append({
                            "domain_col": domain_col,
                            "train_domain_value": domA,
                            "test_domain_value": domB,
                            "direction": f"{domA} → {domB}",
                            "emb_set": emb_set,
                            "rep": rep,
                            "task": task,
                            "split_seed": int(split_seed),
                            "C_selected": float(best_C),
                            "n_train": int(len(ytr)),
                            "n_val": int(len(yva)),
                            "n_test": int(len(yte)),
                            "n_classes": int(n_classes),
                            "test_acc": float(te["acc"]),
                            "test_f1_macro": float(te["f1_macro"]),
                        })

            df = pd.DataFrame(results)
            df.to_csv(RESULTS_CSV, index=False)
            print("[OK] Saved:", RESULTS_CSV)

            if df.empty:
                print("[DONE] No results for:", domain_col, domA, "->", domB, "|", emb_set)
                continue

            summ = (
                df.groupby(["domain_col", "train_domain_value", "test_domain_value", "emb_set", "task", "rep"], as_index=False)
                  .agg(mean_f1=("test_f1_macro", "mean"),
                       std_f1=("test_f1_macro", "std"),
                       n=("test_f1_macro", "count"))
                  .sort_values(["domain_col", "emb_set", "task", "train_domain_value", "test_domain_value", "rep"])
                  .reset_index(drop=True)
            )
            summ.to_csv(SUMMARY_CSV, index=False)
            print("[OK] Saved:", SUMMARY_CSV)

            for task in summ["task"].unique().tolist():
                plot_domain_shift_by_task(
                    summ,
                    task=task,
                    domain_col=domain_col,
                    emb_set=emb_set,
                    out_png=OUT_PLOT / f"domain_shift__{_slug(domain_col)}__{_slug(emb_set)}__{_slug(task)}.png",
                )

            print("[DONE] Outputs:")
            print(" ", OUT_PAIR.resolve())


# ============================================================
# MAIN
# ============================================================
def main():
    for p in [CHAN_PQ, MSI_META_CSV]:
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")

    for emb_set in EMB_SETS:
        if emb_set not in EMB_SET_TO_FILES:
            raise KeyError(f"Unknown emb_set {emb_set}. Valid: {sorted(EMB_SET_TO_FILES.keys())}")

    print("[LOAD] channels parquet:", CHAN_PQ)
    ch_df_full = pd.read_parquet(
        CHAN_PQ,
        columns=[DATASET_COL, TILE_R_COL, TILE_C_COL, CHAN_IDX_COL],
    ).reset_index(drop=True)
    ch_df_full[DATASET_COL] = ch_df_full[DATASET_COL].astype(str)

    print("[LOAD] meta.csv:", MSI_META_CSV)
    meta = pd.read_csv(MSI_META_CSV)
    meta[DATASET_COL] = meta[DATASET_COL].astype(str)

    need = {DATASET_COL} | set(TASKS) | set(DOMAIN_COLS)
    miss = need - set(meta.columns)
    if miss:
        raise KeyError(f"meta.csv missing required columns: {sorted(miss)}")

    agg_cols = list(dict.fromkeys(DOMAIN_COLS + TASKS))
    meta_ds_all = (
        meta.groupby(DATASET_COL, as_index=False)
            .agg({c: (lambda s: s.dropna().iloc[0] if len(s.dropna()) else np.nan) for c in agg_cols})
    )

    # normalize domain columns and configured domain values
    for c in DOMAIN_COLS:
        meta_ds_all[c] = meta_ds_all[c].map(norm_str)
    for k in list(DOMAIN_VALUES.keys()):
        DOMAIN_VALUES[k] = [norm_str(v) for v in DOMAIN_VALUES[k]]

    for domain_col in DOMAIN_COLS:
        if domain_col == "Condition" and "Condition" in TASKS:
            continue
        run_one_domain_col(
            domain_col=domain_col,
            ch_df_full=ch_df_full,
            meta_ds_all=meta_ds_all,
        )


if __name__ == "__main__":
    main()

[LOAD] channels parquet: metaspace_images_dump\channels_with_candidates.parquet
[LOAD] meta.csv: msi-vitmae_REBUILT2_2\embeddings\meta.csv

[DOMAIN] DOMAIN_COL=polarity  TRAIN='positive'  TEST='negative'


Load reps (polarity positive->negative | fused_all):   0%|          | 0/4 [00:00<?, ?it/s]

[INFO] rep=PCA          emb_set=fused_all      rows in domA/domB: 157,847
[INFO] rep=MAE-RandInit emb_set=fused_all      rows in domA/domB: 157,847
[INFO] rep=MAE-ImageNet emb_set=fused_all      rows in domA/domB: 157,847
[INFO] rep=MAE-MSI      emb_set=fused_all      rows in domA/domB: 157,847


Linear probe reps (polarity positive->negative | fused_all):   0%|          | 0/4 [00:00<?, ?it/s]

Tasks (polarity/positive->negative/fused_all/PCA):   0%|          | 0/2 [00:00<?, ?it/s]

...

Tasks (analyzerType/timstof->orbitrap/fused_all/PCA):   0%|          | 0/2 [00:00<?, ?it/s]

Tasks (analyzerType/timstof->orbitrap/fused_all/MAE-RandInit):   0%|          | 0/2 [00:00<?, ?it/s]

Tasks (analyzerType/timstof->orbitrap/fused_all/MAE-ImageNet):   0%|          | 0/2 [00:00<?, ?it/s]

Tasks (analyzerType/timstof->orbitrap/fused_all/MAE-MSI):   0%|          | 0/2 [00:00<?, ?it/s]

[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel\domain_analyzerType\timstof__to__orbitrap\fused_all\results_all.csv
[DONE] No results for: analyzerType timstof -> orbitrap | fused_all


### HMDB

In [ ]:
# ============================================================
# DOMAIN SHIFT (LINEAR PROBE ONLY) — HMDB TAGS (COMPARE EMBEDDING SETS, ONE REP)
#
# ✅ Your request:
#   - Compare EMBEDDING SETS (NOT reps)
#   - Use a single selected REP (default: "MAE-MSI")
#   - Embedding sets (4):
#       1) chan_only            (built from raw channel embeddings_all.npy)
#       2) chan+cls             (load fused_msi__REP.npy)            [cls+chan, no smiles]
#       3) chan+cls+mzpol       (built on-the-fly)                   [1/3 cls + 1/3 chan + 1/3 mzpol]
#       4) chan+cls+smiles      (load fused_all__REP.npy)            [cls+chan+smiles]
#
# ✅ Domain shift protocol:
#   - Train on DOMAIN=A datasets, test on DOMAIN=B datasets.
#   - Inside TRAIN domain only: dataset-grouped TRAIN/VAL split (no leakage).
#   - Explicit DOMAIN_VALUES (no auto top-2).
#   - analyzerType has 3 values -> all pairwise shifts both directions.
#
# ✅ HMDB supports multiple fields in one run:
#   HMDB_FIELDS = ["super_class","class", ...]
#
# ✅ Outputs (per domain pair):
#   spatial_metabolomics_atlas/benchmarks/domain_shift_hmdb_lp_compare_embsets/
#     domain_<DOMAIN_COL>/<A>__to__<B>/
#       results_all.csv
#       summary_meanstd.csv
#       leaderboard_best_by_field.csv
#       plots/domain_shift__hmdb_<field>.png
#
# Notes:
# - This is a HMDB row-level task (rows = channels parquet rows).
# - Dataset-level domain labels come from MSI meta.csv (one per dataset_id).
# - HMDB labels come from HMDB_PQ (cid + hmdb_taxonomy), majority-vote over candidate CIDs.
#
# ============================================================

from __future__ import annotations

from pathlib import Path
import itertools
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning)
warnings.filterwarnings("ignore", module="sklearn")


# ============================================================
# PATHS
# ============================================================
BASE = Path("metaspace_images_dump")
CHAN_PQ = BASE / "channels_with_candidates.parquet"
HMDB_PQ = BASE / "molformer_pubchem_index_enriched_semantic.parquet"  # contains: cid, hmdb_taxonomy

MSI_DIR = Path("msi-vitmae_REBUILT2_2/embeddings")
MSI_META_CSV = MSI_DIR / "meta.csv"   # dataset-level domain labels
MSI_CLS_NPY  = MSI_DIR / "cls_embeddings.npy"

# MAE-MSI channel embeddings (tile_stack)
MSI_CHAN_DIR = Path("msi-vitmae_REBUILT2_2/embeddings_single_channel")
MSI_CHAN_ALL_NPY  = MSI_CHAN_DIR / "embeddings_all.npy"   # (Ntiles, C, D)
MSI_CHAN_ALL_META = MSI_CHAN_DIR / "meta__all.csv"        # has dataset_id,tile_r,tile_c,(split?)

ATLAS = Path("spatial_metabolomics_atlas")
EMB_DIR = ATLAS / "channel_embeddings"

OUT_ROOT = ATLAS / "benchmarks" / "domain_shift_single_channel_hmdb"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
HMDB_CACHE_DIR = OUT_ROOT / "_hmdb_cache"
HMDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# CONFIG
# ============================================================
SEED = 6740

# ✅ Single rep (compare embedding sets)
REP = "MAE-MSI"

# Domain shift settings
DOMAIN_VALUES = {
    "polarity": ["positive", "negative"],
    "organism": ["mus musculus", "homo sapiens"],
    "ionisationSource": ["DESI", "MALDI"],
    "analyzerType": ["FTICR", "Orbitrap", "timsTOF"],
}
DOMAIN_COLS = list(DOMAIN_VALUES.keys())

# Split (inside TRAIN domain only, grouped by dataset_id)
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.20
assert abs((TRAIN_FRAC + VAL_FRAC) - 1.0) < 1e-9

SPLIT_REPEATS = 3
SPLIT_SEEDS = [SEED + 1000 * r for r in range(SPLIT_REPEATS)]

# Linear probe hyperparams (fast)
C_GRID = [1.0]  # keep small for speed
LP_TOL = 1e-3
MAX_EPOCHS_BIN   = 2000
MAX_EPOCHS_MULTI = 2000

# HMDB fields to evaluate (multiple)
HMDB_FIELDS = ["super_class", "class"]
UNKNOWN_LABEL = "unknown"

# Minimums / filters
MIN_DS_PER_CLASS_FOR_DS_SPLIT = 2   # dataset-level strat split inside TRAIN domain
MIN_TRAIN_ROWS_PER_CLASS      = 50  # row-level class count in TRAIN rows
MIN_TRAIN_ROWS_TOTAL          = 500
MIN_TEST_ROWS_TOTAL           = 500

# Optional caps for speed (balanced by dataset)
MAX_TRAIN_ROWS = 30000   # None to disable
MAX_TEST_ROWS  = None    # None to disable

# Channels parquet columns
DATASET_COL   = "dataset_id"
TILE_R_COL    = "tile_r"
TILE_C_COL    = "tile_c"
CHAN_IDX_COL  = "channel_idx"
MZ_COL        = "mz"
CAND_CIDS_COL = "cand_pubchem_cids"


# ============================================================
# PLOT CONFIG (match your earlier domain-shift style)
# ============================================================
# Order and display names for embedding sets
EMB_SET_ORDER = [
    "chan_only",
    "chan+cls",
    "chan+cls+mzpol",
    "chan+cls+smiles",
]
EMB_SET_DISPLAY = {
    "chan_only": "chan_only",
    "chan+cls": "chan+cls",
    "chan+cls+mzpol": "chan+cls+mzpol",
    "chan+cls+smiles": "chan+cls+smiles",
}


# ============================================================
# Small helpers
# ============================================================
def _slug(s: str) -> str:
    s = str(s)
    out = []
    for ch in s:
        out.append(ch if ch.isalnum() else "_")
    out = "".join(out)
    while "__" in out:
        out = out.replace("__", "_")
    out = out.strip("_")
    return out[:120] if out else "x"

def norm_str(x) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip().lower()

def all_ordered_domain_pairs(values: list[str]) -> list[tuple[str, str]]:
    pairs = []
    for a, b in itertools.combinations(values, 2):
        pairs.append((a, b))
        pairs.append((b, a))
    return pairs

def l2_rows(X: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32, order="C")
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)

def standardize_fit(X: np.ndarray):
    mu = X.mean(axis=0, dtype=np.float32).astype(np.float32)
    sd = X.std(axis=0, dtype=np.float32).astype(np.float32)
    sd[sd == 0] = 1.0
    return mu, sd

def standardize_apply(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:
    X = np.array(X, dtype=np.float32, copy=True)
    np.subtract(X, mu, out=X)
    np.divide(X, sd, out=X)
    return X

def _alpha_from_C(C: float) -> float:
    C = float(C)
    if C <= 0:
        C = 1.0
    return 1.0 / C

def fit_sgd_logreg(Xtr_s, ytr, Xte_s, yte, C, seed, max_epochs):
    clf = SGDClassifier(
        loss="log_loss",
        alpha=_alpha_from_C(C),
        penalty="l2",
        max_iter=int(max_epochs),
        tol=float(LP_TOL),
        n_jobs=-1,
        class_weight="balanced",
        random_state=int(seed),
    )
    clf.fit(Xtr_s, ytr)
    yhat = clf.predict(Xte_s)
    return {
        "acc": float(accuracy_score(yte, yhat)),
        "f1_macro": float(f1_score(yte, yhat, average="macro")),
    }

def pick_C_on_val(Xtr_s, ytr, Xva_s, yva, C_grid, seed, max_epochs) -> float:
    best_C, best_f1 = None, -1.0
    for C in C_grid:
        res = fit_sgd_logreg(Xtr_s, ytr, Xva_s, yva, C=C, seed=seed, max_epochs=max_epochs)
        if res["f1_macro"] > best_f1:
            best_f1 = res["f1_macro"]
            best_C = float(C)
    return float(best_C if best_C is not None else float(C_grid[0]))

def cap_rows_balanced_by_dataset(idx: np.ndarray, ds_codes: np.ndarray, max_total: int | None, seed: int) -> np.ndarray:
    idx = np.asarray(idx, dtype=np.int64)
    if max_total is None or len(idx) <= int(max_total):
        return idx
    rng = np.random.default_rng(int(seed))
    codes = ds_codes[idx].astype(np.int32, copy=False)
    uniq = np.unique(codes)
    rng.shuffle(uniq)
    base = int(max_total) // max(len(uniq), 1)
    rem  = int(max_total) - base * len(uniq)

    kept = []
    for code in uniq:
        arr = idx[codes == code].copy()
        rng.shuffle(arr)
        take = base + (1 if rem > 0 else 0)
        if rem > 0:
            rem -= 1
        kept.append(arr[:min(take, len(arr))])
    out = np.concatenate(kept) if kept else idx[:int(max_total)]
    rng.shuffle(out)
    return out[:int(max_total)]


# ============================================================
# HMDB labeling (majority vote over candidate CIDs) — cached ALLROWS
# ============================================================
_CID_RE = re.compile(r"\bCID\s*0*([0-9]+)\b", flags=re.IGNORECASE)

def normalize_cid(x) -> str | None:
    if x is None:
        return None
    s = str(x).strip()
    if not s or s.lower() in {"none", "nan"}:
        return None
    m = _CID_RE.search(s)
    if m:
        return f"CID{int(m.group(1))}"
    if s.isdigit():
        return f"CID{int(s)}"
    return None

def parse_cids_field(cand_pubchem_cids) -> list[str]:
    if cand_pubchem_cids is None:
        return []
    if isinstance(cand_pubchem_cids, (list, tuple, np.ndarray)):
        toks = list(cand_pubchem_cids)
    else:
        s = str(cand_pubchem_cids).strip()
        if not s or s.lower() in {"none", "nan"}:
            return []
        toks = re.split(r"[;,|]\s*|\s+\|\s+|\s*;\s*", s)

    out = []
    for t in toks:
        cid = normalize_cid(t)
        if cid:
            out.append(cid)

    seen, uniq = set(), []
    for cid in out:
        if cid not in seen:
            uniq.append(cid); seen.add(cid)
    return uniq

def _hmdb_tag(tax_str, key: str, default=UNKNOWN_LABEL) -> str:
    if tax_str is None:
        return default
    try:
        if pd.isna(tax_str):
            return default
    except Exception:
        pass

    key_l = key.lower()
    for p in str(tax_str).split(";"):
        p = p.strip()
        if not p:
            continue
        if p.lower().startswith(key_l + ":"):
            v = p.split(":", 1)[1].strip()
            return v if v else default
    return default

def build_cid_to_taxonomy(pq_path: Path) -> dict[str, str]:
    pq_df = pd.read_parquet(pq_path, columns=["cid", "hmdb_taxonomy"])
    pq_df["cid_norm"] = pq_df["cid"].astype(str).map(normalize_cid)
    cid_to_tax: dict[str, str] = {}
    for cid, tax in zip(pq_df["cid_norm"].values, pq_df["hmdb_taxonomy"].values):
        if cid and cid not in cid_to_tax:
            cid_to_tax[cid] = tax
    return cid_to_tax

def majority_vote(tags: list[str]) -> str:
    vc = pd.Series(tags).value_counts()
    return str(vc.index[0])

def label_row_by_hmdb_field_fast(cand_pubchem_cids, field: str, cid_to_tax: dict[str, str]) -> str:
    cids = parse_cids_field(cand_pubchem_cids)
    if not cids:
        return UNKNOWN_LABEL

    tags = []
    for cid in cids:
        tax = cid_to_tax.get(cid)
        if tax is None:
            continue
        v = _hmdb_tag(tax, field, default=UNKNOWN_LABEL)
        if v != UNKNOWN_LABEL:
            tags.append(v)

    if not tags:
        return UNKNOWN_LABEL
    return majority_vote(tags)

def compute_hmdb_labels_for_all_rows_cached(
    ch_df_all: pd.DataFrame,
    cid_to_tax: dict[str, str],
    cache_dir: Path,
    field: str,
) -> np.ndarray:
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_csv = cache_dir / f"hmdb__{field}__ALLROWS.csv"

    if cache_csv.exists():
        df = pd.read_csv(cache_csv)
        m = dict(zip(df["row_id"].astype(np.int64).values, df["label"].astype(str).values))
        labels = np.array([m.get(int(i), UNKNOWN_LABEL) for i in range(len(ch_df_all))], dtype=object)
        print(f"[HMDB] loaded cached ALLROWS {field}: {cache_csv.name}")
        return labels.astype(object).astype(str)

    print(f"[HMDB] computing ALLROWS {field} for N={len(ch_df_all):,} ...")
    cids_col = ch_df_all[CAND_CIDS_COL].values
    labels = np.empty(len(ch_df_all), dtype=object)
    for i, cand_field in enumerate(tqdm(cids_col, desc=f"HMDB {field}", leave=False)):
        labels[i] = label_row_by_hmdb_field_fast(cand_field, field, cid_to_tax)

    pd.DataFrame({"row_id": np.arange(len(ch_df_all), dtype=np.int64), "label": labels.astype(str)}).to_csv(cache_csv, index=False)
    print(f"[HMDB] saved ALLROWS {field}: {cache_csv.name}")
    return labels.astype(object).astype(str)


# ============================================================
# Embedding builders/loaders for the 4 sets (single REP)
# ============================================================
def load_saved(rep: str, kind: str) -> tuple[np.ndarray, np.ndarray]:
    """
    kind in {"all","msi"}
    - all: cls+chan+smiles
    - msi: cls+chan (no smiles)
    """
    if kind == "all":
        Xp = EMB_DIR / f"fused_all__{rep}.npy"
        Ip = EMB_DIR / f"row_ids__all__{rep}.npy"
    elif kind == "msi":
        Xp = EMB_DIR / f"fused_msi__{rep}.npy"
        Ip = EMB_DIR / f"row_ids__msi__{rep}.npy"
    else:
        raise ValueError(kind)

    if not (Xp.exists() and Ip.exists()):
        raise FileNotFoundError(f"Missing fused files:\n  {Xp}\n  {Ip}")
    X = np.load(Xp, mmap_mode="r")
    row_ids = np.asarray(np.load(Ip), dtype=np.int64)
    if len(X) != len(row_ids):
        raise RuntimeError(f"Len mismatch: {Xp.name} X={np.shape(X)} ids={row_ids.shape}")
    return np.asarray(X, dtype=np.float32, order="C"), row_ids

SPLIT_PREFER_ORDER = ("train", "val", "test")

def build_chan_tile_index(meta_df: pd.DataFrame) -> tuple[dict[tuple, int], bool]:
    need = {DATASET_COL, TILE_R_COL, TILE_C_COL}
    miss = need - set(meta_df.columns)
    if miss:
        raise KeyError(f"channel meta missing columns: {sorted(miss)}")

    has_split = "split" in meta_df.columns
    key_to_i: dict[tuple, int] = {}
    for i, rr in meta_df.reset_index(drop=True).iterrows():
        ds = str(rr[DATASET_COL])
        tr = int(rr[TILE_R_COL])
        tc = int(rr[TILE_C_COL])
        if has_split:
            sp = str(rr["split"])
            key = (ds, tr, tc, sp)
        else:
            key = (ds, tr, tc)
        if key not in key_to_i:
            key_to_i[key] = i
    return key_to_i, has_split

def infer_tile_key_for_chan_embeddings(
    row: pd.Series,
    *,
    chan_meta_has_split: bool,
    chan_tile_to_i: dict[tuple, int],
    prefer_order: tuple[str, ...] = SPLIT_PREFER_ORDER,
) -> tuple:
    ds = str(row.get(DATASET_COL))
    tr = int(row.get(TILE_R_COL))
    tc = int(row.get(TILE_C_COL))

    if not chan_meta_has_split:
        return (ds, tr, tc)

    if "split" in row.index and row.get("split") is not None:
        return (ds, tr, tc, str(row.get("split")))

    for sp in prefer_order:
        k = (ds, tr, tc, sp)
        if k in chan_tile_to_i:
            return k

    prefix = (ds, tr, tc)
    for k in chan_tile_to_i.keys():
        if len(k) == 4 and k[:3] == prefix:
            return k

    return (ds, tr, tc, prefer_order[0])

def fetch_chan_embeddings_for_row_ids(
    *,
    row_ids: np.ndarray,
    ch_df_full: pd.DataFrame,
    Z_all: np.ndarray,         # (Ntiles, C, D)
    meta_all: pd.DataFrame,    # tile index
) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns:
      X_out (N_kept, D) float32
      row_ids_kept (N_kept,)
    Drops any rows that cannot be mapped to a tile/channel.
    """
    tile_to_i, has_split = build_chan_tile_index(meta_all)

    D = int(Z_all.shape[2])
    X_out = np.empty((len(row_ids), D), dtype=np.float32)
    ok = np.ones(len(row_ids), dtype=bool)

    for j, rid in enumerate(tqdm(row_ids, desc="build chan_only", leave=False)):
        r = ch_df_full.iloc[int(rid)]
        k = int(r[CHAN_IDX_COL])

        key_tile = infer_tile_key_for_chan_embeddings(
            r, chan_meta_has_split=has_split, chan_tile_to_i=tile_to_i
        )
        ti = tile_to_i.get(key_tile)
        if ti is None or not (0 <= k < Z_all.shape[1]):
            ok[j] = False
            continue
        X_out[j, :] = np.asarray(Z_all[ti, k, :], dtype=np.float32)

    if not np.all(ok):
        row_ids_kept = row_ids[ok]
        X_out = X_out[ok]
        return X_out, row_ids_kept
    return X_out, row_ids


# ============================================================
# MZPOL (same as your earlier definition)
# ============================================================
def infer_meta_polarity_column(meta: pd.DataFrame) -> str:
    cols = list(meta.columns)
    low = {c.lower(): c for c in cols}

    preferred = [
        "polarity",
        "ion_polarity",
        "ionisation_polarity",
        "ionization_polarity",
        "ionisation",
        "ionization",
    ]
    for k in preferred:
        if k in low:
            return low[k]

    for c in cols:
        if "pol" in c.lower():
            return c

    raise KeyError(
        "Could not find a polarity column in meta.csv. "
        "Rename/add a column containing 'pol' or set POL_COL manually."
    )

def pol_to_scalar(p) -> float:
    if p is None:
        return 0.0
    s = str(p).strip().lower()
    if not s or s in {"nan", "none"}:
        return 0.0
    if "neg" in s or s in {"-", "-1", "negative"}:
        return -1.0
    if "pos" in s or s in {"+", "+1", "1", "positive"}:
        return 1.0
    try:
        v = float(s)
        if v > 0:
            return 1.0
        if v < 0:
            return -1.0
        return 0.0
    except Exception:
        return 0.0

MZPOL_OUT_DIM       = 768
MZPOL_FOURIER_BANDS = 16
MZPOL_FOURIER_SCALE = 10.0
MZPOL_USE_LOG_MZ    = True

def fourier_mz_features(mz: np.ndarray, bands: int, scale: float, use_log: bool) -> np.ndarray:
    mz = np.asarray(mz, dtype=np.float32)
    mz2 = np.where(np.isfinite(mz) & (mz > 0), mz, np.nan).astype(np.float32)
    x = np.log(mz2) if use_log else mz2

    if np.any(~np.isfinite(x)):
        med = np.nanmedian(x)
        if not np.isfinite(med):
            med = 0.0
        x = np.where(np.isfinite(x), x, med).astype(np.float32)

    ks = (2.0 ** np.arange(int(bands), dtype=np.float32))[None, :]
    arg = (x[:, None] * ks) / float(scale)
    sin = np.sin(arg)
    cos = np.cos(arg)
    return np.concatenate([sin, cos], axis=1).astype(np.float32)

def build_mzpol_embedding_768(mz: np.ndarray, pol_scalar: np.ndarray) -> np.ndarray:
    mz_feat = fourier_mz_features(
        mz=mz,
        bands=int(MZPOL_FOURIER_BANDS),
        scale=float(MZPOL_FOURIER_SCALE),
        use_log=bool(MZPOL_USE_LOG_MZ),
    )
    pol_scalar = np.asarray(pol_scalar, dtype=np.float32).reshape(-1, 1)
    bias = np.ones((len(mz_feat), 1), dtype=np.float32)
    feats = np.concatenate([mz_feat, pol_scalar, bias], axis=1).astype(np.float32)

    rng = np.random.default_rng(SEED)
    W = rng.standard_normal((feats.shape[1], int(MZPOL_OUT_DIM)), dtype=np.float32)
    Z = feats @ W
    return l2_rows(Z)

def build_tile_key_to_pol(msi_meta: pd.DataFrame) -> dict[tuple, float]:
    POL_COL = infer_meta_polarity_column(msi_meta)
    need = {DATASET_COL, TILE_R_COL, TILE_C_COL, POL_COL}
    miss = need - set(msi_meta.columns)
    if miss:
        raise KeyError(f"MSI meta.csv missing required columns: {sorted(miss)}")

    key_to_pol: dict[tuple, float] = {}
    for _, r in msi_meta.reset_index(drop=True).iterrows():
        key = (str(r[DATASET_COL]), int(r[TILE_R_COL]), int(r[TILE_C_COL]))
        if key not in key_to_pol:
            key_to_pol[key] = float(pol_to_scalar(r[POL_COL]))
    return key_to_pol

def compute_mzpol_for_row_ids(
    *,
    row_ids: np.ndarray,
    ch_df_full: pd.DataFrame,
    key_to_pol: dict[tuple, float],
) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns:
      Z_mzpol (N_kept, 768)
      row_ids_kept (N_kept,)
    Drops rows where mz not finite.
    """
    mz_vec = ch_df_full.iloc[row_ids][MZ_COL].to_numpy()
    ds_vec = ch_df_full.iloc[row_ids][DATASET_COL].astype(str).to_numpy()
    tr_vec = ch_df_full.iloc[row_ids][TILE_R_COL].astype(int).to_numpy()
    tc_vec = ch_df_full.iloc[row_ids][TILE_C_COL].astype(int).to_numpy()

    pol_s = np.zeros(len(row_ids), dtype=np.float32)
    for j in range(len(row_ids)):
        key = (str(ds_vec[j]), int(tr_vec[j]), int(tc_vec[j]))
        pol_s[j] = float(key_to_pol.get(key, 0.0))

    ok = np.isfinite(mz_vec)
    row_ids_kept = row_ids[ok]
    mz_kept = mz_vec[ok]
    pol_kept = pol_s[ok]

    if len(row_ids_kept) == 0:
        return np.empty((0, 768), dtype=np.float32), np.empty((0,), dtype=np.int64)

    Z_mzpol = build_mzpol_embedding_768(mz=mz_kept, pol_scalar=pol_kept).astype(np.float32, copy=False)
    return Z_mzpol, row_ids_kept


# ============================================================
# Split helper (dataset-grouped split within TRAIN domain only)
# ============================================================
def stratified_group_split_train_val_dataset_ids(ds_ids, ds_labels, seed, val_frac):
    ds_ids = np.asarray(ds_ids).astype(str)
    ds_labels = np.asarray(ds_labels)

    if len(ds_ids) != len(ds_labels):
        raise ValueError("ds_ids and ds_labels must have same length")
    if len(np.unique(ds_labels)) < 2:
        raise ValueError("Need >=2 classes at dataset level for stratified split")

    vc = pd.Series(ds_labels).value_counts()
    if int(vc.min()) < 2:
        raise ValueError("Some dataset-level class has <2 datasets (cannot stratify).")

    idx_all = np.arange(len(ds_ids))
    sss = StratifiedShuffleSplit(n_splits=1, test_size=float(val_frac), random_state=int(seed))
    idx_tr, idx_va = next(sss.split(idx_all, ds_labels))
    return ds_ids[idx_tr], ds_ids[idx_va]


# ============================================================
# Plotting (COMPARE EMB SETS, one rep) — match earlier style
# ============================================================
def plot_domain_shift_by_field(summary_df: pd.DataFrame, *, field: str, domain_col: str, out_png: Path):
    sub = summary_df[(summary_df["field"] == field) & (summary_df["domain_col"] == domain_col)].copy()
    if sub.empty:
        return

    # stable direction order (optional but nice)
    dir_df = sub[["train_domain_value", "test_domain_value"]].drop_duplicates()
    dirs = dir_df.apply(lambda r: f"{r['train_domain_value']} → {r['test_domain_value']}", axis=1).tolist()

    emb_sets = [e for e in EMB_SET_ORDER if e in sub["emb_set"].unique().tolist()]

    mean = {(e, d): np.nan for e in emb_sets for d in dirs}
    std  = {(e, d): 0.0   for e in emb_sets for d in dirs}

    # --- fill grids
    for _, rr in sub.iterrows():
        dlab = f"{rr['train_domain_value']} → {rr['test_domain_value']}"
        emb  = str(rr["emb_set"])

        key = (emb, dlab)
        if key not in mean:
            continue

        m = rr["mean_f1"]
        s = rr["std_f1"]
        mean[key] = float(m) if np.isfinite(m) else np.nan
        std[key]  = float(s) if np.isfinite(s) else 0.0

    x = np.arange(len(dirs))
    width = 0.8 / max(len(emb_sets), 1)

    plt.figure(figsize=(max(10.5, 1.8 * len(dirs)), 4.4))
    for i, emb in enumerate(emb_sets):
        vals = [mean[(emb, d)] for d in dirs]
        errs = [std[(emb, d)]  for d in dirs]

        xpos = x - 0.4 + (i + 0.5) * width
        plt.bar(xpos, vals, width=width, label=EMB_SET_DISPLAY.get(emb, emb))
        plt.errorbar(xpos, vals, yerr=errs, fmt="none", capsize=3, c="black", lw=1)

    plt.xticks(x, dirs, rotation=0)
    plt.ylabel("Test Macro-F1 (mean ± std)")
    plt.title(f"Domain shift — HMDB:{field}\nDOMAIN_COL={domain_col} | REP={REP}")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.savefig(out_png, dpi=300)
    plt.close()
    print("[OK] Saved plot:", out_png)


# ============================================================
# Core eval for one (DOMAIN_COL, A->B), one embedding set, one split_seed
# ============================================================
def evaluate_one_embset_one_pair_one_seed(
    *,
    emb_set: str,
    X: np.ndarray,
    row_ids: np.ndarray,
    ch_df_full: pd.DataFrame,
    ds_domain_map: dict[str, str],
    domain_col: str,
    A: str,
    B: str,
    hmdb_labels_allrows_by_field: dict[str, np.ndarray],
    split_seed: int,
) -> list[dict]:
    """
    Returns rows (one per HMDB field) for a single split_seed.
    """
    ds_rows = ch_df_full.iloc[row_ids][DATASET_COL].astype(str).to_numpy()
    dom_rows = np.array([ds_domain_map.get(d, "") for d in ds_rows], dtype=object)

    m_pair = (dom_rows == A) | (dom_rows == B)
    if not np.any(m_pair):
        return []

    Xp = np.asarray(X[m_pair], dtype=np.float32, order="C")
    row_ids_p = row_ids[m_pair]
    ds_rows_p = ds_rows[m_pair]
    dom_rows_p = dom_rows[m_pair].astype(str)

    # dataset codes for balancing caps
    ds_codes, ds_uniques = pd.factorize(ds_rows_p, sort=False)
    ds_codes = ds_codes.astype(np.int32, copy=False)

    # pool indices
    is_train_dom = (dom_rows_p == A)
    is_test_dom  = (dom_rows_p == B)
    if (not np.any(is_train_dom)) or (not np.any(is_test_dom)):
        return []

    idx_trainpool_all = np.where(is_train_dom)[0].astype(np.int64)
    idx_test_all      = np.where(is_test_dom)[0].astype(np.int64)

    idx_trainpool_all = cap_rows_balanced_by_dataset(
        idx_trainpool_all, ds_codes, MAX_TRAIN_ROWS,
        seed=int(split_seed) + 11 + (hash((emb_set, domain_col, A, B, "traincap")) % 997),
    )
    idx_test_all = cap_rows_balanced_by_dataset(
        idx_test_all, ds_codes, MAX_TEST_ROWS,
        seed=int(split_seed) + 17 + (hash((emb_set, domain_col, A, B, "testcap")) % 997),
    )

    if len(idx_trainpool_all) < MIN_TRAIN_ROWS_TOTAL or len(idx_test_all) < MIN_TEST_ROWS_TOTAL:
        return []

    out_rows: list[dict] = []

    for field in HMDB_FIELDS:
        y_all = hmdb_labels_allrows_by_field[field]
        y_rows = y_all[row_ids_p].astype(str)

        # known labels
        m_known = (y_rows != UNKNOWN_LABEL) & (y_rows != "") & (y_rows != "nan")
        if not np.any(m_known):
            continue

        idx_trainpool = idx_trainpool_all[m_known[idx_trainpool_all]]
        idx_test      = idx_test_all[m_known[idx_test_all]]
        if len(idx_trainpool) < MIN_TRAIN_ROWS_TOTAL or len(idx_test) < MIN_TEST_ROWS_TOTAL:
            continue

        y_train_raw = y_rows[idx_trainpool]
        y_test_raw  = y_rows[idx_test]

        # keep only shared labels (appear on BOTH sides)
        shared = set(pd.unique(y_train_raw)) & set(pd.unique(y_test_raw))
        if len(shared) < 2:
            continue

        m_shared_tr = np.array([yy in shared for yy in y_train_raw], dtype=bool)
        m_shared_te = np.array([yy in shared for yy in y_test_raw], dtype=bool)
        idx_trainpool = idx_trainpool[m_shared_tr]
        idx_test      = idx_test[m_shared_te]
        y_train_raw = y_rows[idx_trainpool]
        y_test_raw  = y_rows[idx_test]

        # train viability: >= MIN_TRAIN_ROWS_PER_CLASS
        vc_tr = pd.Series(y_train_raw).value_counts()
        ok_classes = vc_tr[vc_tr >= int(MIN_TRAIN_ROWS_PER_CLASS)].index.astype(str).tolist()
        if len(ok_classes) < 2:
            continue

        m_ok_tr = np.array([yy in ok_classes for yy in y_train_raw], dtype=bool)
        m_ok_te = np.array([yy in ok_classes for yy in y_test_raw], dtype=bool)
        idx_trainpool = idx_trainpool[m_ok_tr]
        idx_test      = idx_test[m_ok_te]
        y_train_raw = y_rows[idx_trainpool]
        y_test_raw  = y_rows[idx_test]

        if len(pd.unique(y_train_raw)) < 2 or len(pd.unique(y_test_raw)) < 2:
            continue

        # dataset-level labels inside TRAIN domain for stratified dataset split
        df_split = pd.DataFrame({"dataset_id": ds_rows_p[idx_trainpool], "y": y_train_raw})
        ds_mode = (
            df_split.groupby("dataset_id")["y"]
                    .agg(lambda s: str(pd.Series(s).value_counts().index[0]))
                    .reset_index()
        )
        vc_ds = ds_mode["y"].value_counts()
        ok_ds = vc_ds[vc_ds >= int(MIN_DS_PER_CLASS_FOR_DS_SPLIT)].index.astype(str).tolist()
        ds_mode = ds_mode[ds_mode["y"].isin(ok_ds)].reset_index(drop=True)
        if ds_mode["y"].nunique() < 2:
            continue

        ds_ids = ds_mode["dataset_id"].astype(str).values
        y_ds_codes = pd.Categorical(ds_mode["y"].astype(str)).codes.astype(np.int64)

        try:
            ds_tr, ds_va = stratified_group_split_train_val_dataset_ids(
                ds_ids=ds_ids,
                ds_labels=y_ds_codes,
                seed=int(split_seed) + 101 + (hash((domain_col, A, B, field, emb_set)) % 997),
                val_frac=VAL_FRAC,
            )
        except Exception:
            continue

        ds_tr_set = set(map(str, ds_tr))
        ds_va_set = set(map(str, ds_va))

        ds_trainpool = ds_rows_p[idx_trainpool]
        m_tr = np.array([d in ds_tr_set for d in ds_trainpool], dtype=bool)
        m_va = np.array([d in ds_va_set for d in ds_trainpool], dtype=bool)
        tr_rel = np.where(m_tr)[0]
        va_rel = np.where(m_va)[0]
        if len(tr_rel) < 50 or len(va_rel) < 20:
            continue

        # stable category order (freq in train, then name)
        vc = pd.Series(y_train_raw[tr_rel]).value_counts()
        cats = sorted(vc.index.astype(str).tolist(), key=lambda c: (-int(vc.get(c, 0)), str(c)))

        y_trainpool_codes = pd.Categorical(pd.Series(y_train_raw), categories=cats, ordered=True).codes.astype(np.int64)
        y_test_codes      = pd.Categorical(pd.Series(y_test_raw),  categories=cats, ordered=True).codes.astype(np.int64)
        if (y_trainpool_codes < 0).any() or (y_test_codes < 0).any():
            continue

        Xtr = Xp[idx_trainpool[tr_rel]]
        ytr = y_trainpool_codes[tr_rel]
        Xva = Xp[idx_trainpool[va_rel]]
        yva = y_trainpool_codes[va_rel]
        Xte = Xp[idx_test]
        yte = y_test_codes

        n_classes = int(len(np.unique(ytr)))
        max_epochs = MAX_EPOCHS_BIN if n_classes == 2 else MAX_EPOCHS_MULTI

        mu, sd = standardize_fit(Xtr)
        Xtr_s = standardize_apply(Xtr, mu, sd)
        Xva_s = standardize_apply(Xva, mu, sd)
        Xte_s = standardize_apply(Xte, mu, sd)

        best_C = pick_C_on_val(
            Xtr_s, ytr, Xva_s, yva,
            C_grid=C_GRID,
            seed=int(split_seed) + 13 + (hash((emb_set, domain_col, A, B, field)) % 997),
            max_epochs=max_epochs,
        )
        te = fit_sgd_logreg(
            Xtr_s, ytr, Xte_s, yte,
            C=best_C,
            seed=int(split_seed) + 77 + (hash((emb_set, domain_col, A, B, field)) % 997),
            max_epochs=max_epochs,
        )

        out_rows.append({
            "domain_col": domain_col,
            "train_domain_value": A,
            "test_domain_value": B,
            "direction": f"{A} → {B}",
            "rep": REP,
            "emb_set": emb_set,
            "field": field,
            "split_seed": int(split_seed),
            "C_selected": float(best_C),
            "n_train": int(len(ytr)),
            "n_val": int(len(yva)),
            "n_test": int(len(yte)),
            "n_classes": int(n_classes),
            "test_acc": float(te["acc"]),
            "test_f1_macro": float(te["f1_macro"]),
        })

    return out_rows


# ============================================================
# Summaries
# ============================================================
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    summ = (
        df.groupby(["domain_col", "train_domain_value", "test_domain_value", "field", "rep", "emb_set"], as_index=False)
          .agg(
              mean_f1=("test_f1_macro", "mean"),
              std_f1=("test_f1_macro", "std"),
              n=("test_f1_macro", "count"),
              mean_acc=("test_acc", "mean"),
          )
          .sort_values(["field", "domain_col", "train_domain_value", "test_domain_value", "mean_f1"], ascending=[True, True, True, True, False])
          .reset_index(drop=True)
    )
    return summ

def leaderboard_best_by_field(summary_df: pd.DataFrame) -> pd.DataFrame:
    if summary_df.empty:
        return summary_df
    best = (
        summary_df.sort_values(["field", "mean_f1"], ascending=[True, False])
                  .groupby("field", as_index=False)
                  .head(1)
                  .reset_index(drop=True)
    )
    best.insert(0, "section", "BEST_PER_FIELD")
    all2 = summary_df.copy()
    all2.insert(0, "section", "ALL")
    return pd.concat([best, all2], axis=0, ignore_index=True)


# ============================================================
# BUILD THE 4 EMBEDDING SETS ONCE (for the selected REP)
# ============================================================
def build_embedding_sets_for_rep(
    *,
    rep: str,
    ch_df_full: pd.DataFrame,
    Z_chan_all: np.ndarray,
    meta_chan_all: pd.DataFrame,
    Z_cls_all: np.ndarray,
    msi_meta: pd.DataFrame,
) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """
    Returns: emb_set -> (X, row_ids)
      - chan_only         : built using row_ids from fused_msi (cls+chan) as universe
      - chan+cls          : fused_msi
      - chan+cls+smiles   : fused_all
      - chan+cls+mzpol    : built on row_ids from fused_msi as universe, then intersect with map-able rows
    """
    out: dict[str, tuple[np.ndarray, np.ndarray]] = {}

    # load saved:
    X_msi, row_msi = load_saved(rep, kind="msi")  # cls+chan
    X_all, row_all = load_saved(rep, kind="all")  # cls+chan+smiles

    out["chan+cls"] = (X_msi, row_msi)
    out["chan+cls+smiles"] = (X_all, row_all)

    # chan_only baseline uses SAME row universe as chan+cls (row_msi)
    X_chan_only, row_chan_only = fetch_chan_embeddings_for_row_ids(
        row_ids=row_msi,
        ch_df_full=ch_df_full,
        Z_all=Z_chan_all,
        meta_all=meta_chan_all,
    )
    out["chan_only"] = (X_chan_only, row_chan_only)

    # chan+cls+mzpol: (1/3 cls + 1/3 chan + 1/3 mzpol) on row_msi universe
    key_to_pol = build_tile_key_to_pol(msi_meta)

    # mzpol needs mz to be finite; will drop some rows
    Z_mzpol, row_mz = compute_mzpol_for_row_ids(row_ids=row_msi, ch_df_full=ch_df_full, key_to_pol=key_to_pol)
    if len(row_mz) > 0:
        # subset cls embeddings to those row_mz by tile-key mapping (using MSI meta.csv indexing)
        # We'll map row_id -> tile (ds,tr,tc) -> cls index from msi_meta row order (same as Z_cls_all)
        # Build key->i for CLS indexing:
        need = {DATASET_COL, TILE_R_COL, TILE_C_COL}
        miss = need - set(msi_meta.columns)
        if miss:
            raise KeyError(f"MSI meta.csv missing required columns for CLS mapping: {sorted(miss)}")

        key_to_cls_i: dict[tuple, int] = {}
        for i, r in msi_meta.reset_index(drop=True).iterrows():
            key = (str(r[DATASET_COL]), int(r[TILE_R_COL]), int(r[TILE_C_COL]))
            if key not in key_to_cls_i:
                key_to_cls_i[key] = int(i)

        ds_vec = ch_df_full.iloc[row_mz][DATASET_COL].astype(str).to_numpy()
        tr_vec = ch_df_full.iloc[row_mz][TILE_R_COL].astype(int).to_numpy()
        tc_vec = ch_df_full.iloc[row_mz][TILE_C_COL].astype(int).to_numpy()

        cls_idx = np.full(len(row_mz), -1, dtype=np.int64)
        for j in range(len(row_mz)):
            key = (str(ds_vec[j]), int(tr_vec[j]), int(tc_vec[j]))
            cls_idx[j] = int(key_to_cls_i.get(key, -1))

        ok_cls = cls_idx >= 0
        row_mz2 = row_mz[ok_cls]
        cls_idx2 = cls_idx[ok_cls]
        Z_mzpol2 = Z_mzpol[ok_cls]

        if len(row_mz2) > 0:
            # fetch channel embeddings for row_mz2 (may drop more)
            X_chan_sub, row_chan_kept = fetch_chan_embeddings_for_row_ids(
                row_ids=row_mz2,
                ch_df_full=ch_df_full,
                Z_all=Z_chan_all,
                meta_all=meta_chan_all,
            )

            if len(row_chan_kept) > 0:
                # align mzpol + cls to row_chan_kept (intersection)
                pos = {int(r): i for i, r in enumerate(row_mz2)}
                take = np.array([pos[int(r)] for r in row_chan_kept], dtype=np.int64)

                Z_mzpol_kept = Z_mzpol2[take]
                cls_idx_kept = cls_idx2[take]

                Z_cls_kept  = l2_rows(Z_cls_all[cls_idx_kept].astype(np.float32, copy=False))
                Z_chan_kept = l2_rows(X_chan_sub)

                X_ccm = l2_rows((1.0/3.0) * Z_cls_kept + (1.0/3.0) * Z_chan_kept + (1.0/3.0) * Z_mzpol_kept).astype(np.float32, copy=False)
                out["chan+cls+mzpol"] = (X_ccm, row_chan_kept)

    # ensure deterministic order keys exist
    for k in EMB_SET_ORDER:
        if k not in out:
            print(f"[WARN] emb_set '{k}' could not be built/loaded (will be skipped).")

    return out


# ============================================================
# MAIN
# ============================================================
def main():
    # --- sanity
    for p in [CHAN_PQ, HMDB_PQ, MSI_META_CSV, MSI_CLS_NPY, MSI_CHAN_ALL_NPY, MSI_CHAN_ALL_META]:
        if not p.exists():
            raise FileNotFoundError(f"Missing: {p}")

    # normalize configured domain values
    for k in list(DOMAIN_VALUES.keys()):
        DOMAIN_VALUES[k] = [norm_str(v) for v in DOMAIN_VALUES[k]]

    # --- load channels parquet (need: dataset_id,tile_r,tile_c,channel_idx,mz,cand_pubchem_cids)
    print("[LOAD] channels parquet:", CHAN_PQ)
    ch_df_full = pd.read_parquet(
        CHAN_PQ,
        columns=[DATASET_COL, TILE_R_COL, TILE_C_COL, CHAN_IDX_COL, MZ_COL, CAND_CIDS_COL],
    ).reset_index(drop=True)
    ch_df_full[DATASET_COL] = ch_df_full[DATASET_COL].astype(str)

    # --- load MSI meta.csv and derive dataset-level domain maps
    print("[LOAD] meta.csv:", MSI_META_CSV)
    meta = pd.read_csv(MSI_META_CSV)
    meta[DATASET_COL] = meta[DATASET_COL].astype(str)

    need = {DATASET_COL} | set(DOMAIN_COLS)
    miss = need - set(meta.columns)
    if miss:
        raise KeyError(f"meta.csv missing required columns: {sorted(miss)}")

    meta_ds = (
        meta.groupby(DATASET_COL, as_index=False)
            .agg({c: (lambda s: s.dropna().iloc[0] if len(s.dropna()) else np.nan) for c in DOMAIN_COLS})
    )
    for c in DOMAIN_COLS:
        meta_ds[c] = meta_ds[c].map(norm_str)

    ds_domain_maps: dict[str, dict[str, str]] = {}
    for c in DOMAIN_COLS:
        ds_domain_maps[c] = dict(zip(meta_ds[DATASET_COL].astype(str).values, meta_ds[c].astype(str).values))

    # --- HMDB label cache
    print("[LOAD] building CID->HMDB taxonomy map ...")
    cid_to_tax = build_cid_to_taxonomy(HMDB_PQ)
    print(f"[INFO] CID->HMDB taxonomy entries: {len(cid_to_tax):,}")

    hmdb_labels_allrows_by_field: dict[str, np.ndarray] = {}
    for field in HMDB_FIELDS:
        hmdb_labels_allrows_by_field[field] = compute_hmdb_labels_for_all_rows_cached(
            ch_df_all=ch_df_full,
            cid_to_tax=cid_to_tax,
            cache_dir=HMDB_CACHE_DIR,
            field=field,
        )

    # --- load CLS + chan embeddings sources (for building chan_only + mzpol variant)
    print("[LOAD] CLS embeddings:", MSI_CLS_NPY)
    Z_cls_all = np.load(MSI_CLS_NPY, mmap_mode="r")
    if Z_cls_all.ndim != 2 or Z_cls_all.shape[1] != 768:
        raise RuntimeError(f"Expected CLS (N_tiles,768), got {Z_cls_all.shape}")

    print("[LOAD] MAE-MSI channel embeddings_all + meta__all")
    Z_chan_all = np.load(MSI_CHAN_ALL_NPY, mmap_mode="r")
    meta_chan_all = pd.read_csv(MSI_CHAN_ALL_META)

    # --- build/load the 4 embedding sets (single rep)
    print(f"\n[BUILD] embedding sets for REP={REP} ...")
    emb_sets_data = build_embedding_sets_for_rep(
        rep=REP,
        ch_df_full=ch_df_full,
        Z_chan_all=Z_chan_all,
        meta_chan_all=meta_chan_all,
        Z_cls_all=Z_cls_all,
        msi_meta=meta,  # tile-level meta.csv
    )
    print("[OK] available emb_sets:", [k for k in EMB_SET_ORDER if k in emb_sets_data])

    # --- run domain shift
    for domain_col in DOMAIN_COLS:
        values = DOMAIN_VALUES[domain_col]
        pairs = all_ordered_domain_pairs(values)

        out_domain = OUT_ROOT / f"domain_{_slug(domain_col)}"
        out_domain.mkdir(parents=True, exist_ok=True)

        ds_domain_map = ds_domain_maps[domain_col]

        for A, B in pairs:
            print(f"\n[DOMAIN] {domain_col}: TRAIN='{A}'  TEST='{B}'")
            out_pair = out_domain / f"{_slug(A)}__to__{_slug(B)}"
            (out_pair / "plots").mkdir(parents=True, exist_ok=True)

            all_rows: list[dict] = []

            # loop over embedding sets (compare these)
            for emb_set in tqdm(EMB_SET_ORDER, desc="emb_sets", leave=False):
                if emb_set not in emb_sets_data:
                    continue

                X, row_ids = emb_sets_data[emb_set]
                # keep as saved/built; (they are l2 already) but ensure float32 contiguous
                Xn = np.asarray(X, dtype=np.float32, order="C")
                row_ids = np.asarray(row_ids, dtype=np.int64)

                for split_seed in SPLIT_SEEDS:
                    rows = evaluate_one_embset_one_pair_one_seed(
                        emb_set=emb_set,
                        X=Xn,
                        row_ids=row_ids,
                        ch_df_full=ch_df_full,
                        ds_domain_map=ds_domain_map,
                        domain_col=domain_col,
                        A=A,
                        B=B,
                        hmdb_labels_allrows_by_field=hmdb_labels_allrows_by_field,
                        split_seed=int(split_seed),
                    )
                    all_rows.extend(rows)

            df = pd.DataFrame(all_rows)
            df.to_csv(out_pair / "results_all.csv", index=False)
            print("[OK] Saved:", out_pair / "results_all.csv")

            summ = summarize(df) if not df.empty else pd.DataFrame()
            summ.to_csv(out_pair / "summary_meanstd.csv", index=False)
            print("[OK] Saved:", out_pair / "summary_meanstd.csv")

            lead = leaderboard_best_by_field(summ) if not summ.empty else pd.DataFrame()
            lead.to_csv(out_pair / "leaderboard_best_by_field.csv", index=False)
            print("[OK] Saved:", out_pair / "leaderboard_best_by_field.csv")

            # plots per field (compare emb_sets)
            if not summ.empty:
                for field in HMDB_FIELDS:
                    plot_domain_shift_by_field(
                        summ,
                        field=field,
                        domain_col=domain_col,
                        out_png=out_pair / "plots" / f"domain_shift__hmdb_{_slug(field)}.png",
                    )

            print("[DONE] Pair outputs:", out_pair.resolve())

    print("\n[DONE] All outputs under:")
    print(" ", OUT_ROOT.resolve())


if __name__ == "__main__":
    main()

[LOAD] channels parquet: metaspace_images_dump\channels_with_candidates.parquet
[LOAD] meta.csv: msi-vitmae_REBUILT2_2\embeddings\meta.csv
[LOAD] building CID->HMDB taxonomy map ...
[INFO] CID->HMDB taxonomy entries: 30,478
[HMDB] loaded cached ALLROWS super_class: hmdb__super_class__ALLROWS.csv
[HMDB] loaded cached ALLROWS class: hmdb__class__ALLROWS.csv
[LOAD] CLS embeddings: msi-vitmae_REBUILT2_2\embeddings\cls_embeddings.npy
[LOAD] MAE-MSI channel embeddings_all + meta__all

[BUILD] embedding sets for REP=MAE-MSI ...


build chan_only:   0%|          | 0/157847 [00:00<?, ?it/s]

build chan_only:   0%|          | 0/157847 [00:00<?, ?it/s]

[OK] available emb_sets: ['chan_only', 'chan+cls', 'chan+cls+mzpol', 'chan+cls+smiles']

[DOMAIN] polarity: TRAIN='positive'  TEST='negative'


emb_sets:   0%|          | 0/4 [00:00<?, ?it/s]

...

[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__fticr\results_all.csv
[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__fticr\summary_meanstd.csv
[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__fticr\leaderboard_best_by_field.csv
[DONE] Pair outputs: \\bme-data2.ad.gatech.edu\labs6\coskun-lab\Efe\MSI Foundation Model\spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__fticr

[DOMAIN] analyzerType: TRAIN='orbitrap'  TEST='timstof'


emb_sets:   0%|          | 0/4 [00:00<?, ?it/s]

[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\orbitrap__to__timstof\results_all.csv
[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\orbitrap__to__timstof\summary_meanstd.csv
[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\orbitrap__to__timstof\leaderboard_best_by_field.csv
[DONE] Pair outputs: \\bme-data2.ad.gatech.edu\labs6\coskun-lab\Efe\MSI Foundation Model\spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\orbitrap__to__timstof

[DOMAIN] analyzerType: TRAIN='timstof'  TEST='orbitrap'


emb_sets:   0%|          | 0/4 [00:00<?, ?it/s]

[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__orbitrap\results_all.csv
[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__orbitrap\summary_meanstd.csv
[OK] Saved: spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__orbitrap\leaderboard_best_by_field.csv
[DONE] Pair outputs: \\bme-data2.ad.gatech.edu\labs6\coskun-lab\Efe\MSI Foundation Model\spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets\domain_analyzerType\timstof__to__orbitrap

[DONE] All outputs under:
  \\bme-data2.ad.gatech.edu\labs6\coskun-lab\Efe\MSI Foundation Model\spatial_metabolomics_atlas\benchmarks\domain_shift_single_channel_hmdb_embedding_sets
